# Nome: Davi dos Santos Mattos
# DRE: 119133049

In [2]:
import numpy as np

def f(X):
    return 2*X[0]**2 + X[1]**2 + 2*X[0]*X[1] + X[0] - 2*X[1] + 3

In [3]:
import jax.numpy as jnp
from jax import grad

def f(X):
    return 2*X[0]**2 + X[1]**2 + 2*X[0]*X[1] + X[0] - 2*X[1] + 3

def grad_f(x_zero):
  grad_f = grad(f)
  return grad_f(jnp.array(x_zero))

result = np.array(grad_f([0.,0.]))
print(result)

[ 1. -2.]


In [4]:
import sympy as sp

x1, x2 = sp.symbols('x1 x2')
f = 2*x1**2 + x2**2 + 2*x1*x2 + x1 - 2*x2 + 3

grad = [sp.diff(f, var) for var in (x1, x2)]
print(grad)

[4*x1 + 2*x2 + 1, 2*x1 + 2*x2 - 2]


In [5]:
import sympy as sp

x1, x2 = sp.symbols('x1 x2')
f = 2*x1**2 + x2**2 + 2*x1*x2 + x1 - 2*x2 + 3

x0 = {x1: -0.75, x2: 1.5}
grad = [sp.diff(f, var) for var in (x1, x2)]
g0 = [g.subs(x0) for g in grad]
print("Gradiente em x0:", g0)

Gradiente em x0: [1.00000000000000, -0.500000000000000]


In [6]:
alpha = sp.symbols('alpha')

x1_alpha = x0[x1] - alpha * g0[0]
x2_alpha = x0[x2] - alpha * g0[1]

print("x1(alpha):", x1_alpha)
print("x2(alpha):", x2_alpha)

x1(alpha): -1.0*alpha - 0.75
x2(alpha): 0.5*alpha + 1.5


In [7]:
f_alpha = f.subs({
    x1: x1_alpha,
    x2: x2_alpha
})

print("f(alpha):")
sp.expand(f_alpha)

f(alpha):


1.25*alpha**2 - 1.25*alpha + 0.375

In [8]:
df_alpha = sp.diff(f_alpha, alpha)
alpha_star = sp.solve(df_alpha, alpha)

In [9]:
df_alpha

2.5*alpha - 1.25

In [10]:
alpha_star

[0.500000000000000]

In [11]:
x1_new = x0[x2] - alpha_star[0] * g0[1]
x1_new

1.75000000000000

# Gradiente Analítico

In [ ]:
import numpy as np
import sympy as sp
import math

x1, x2, alpha = sp.symbols('x1 x2 alpha')


def f():
    x1, x2 = sp.symbols('x1 x2')
    f = 2*x1**2 + x2**2 + 2*x1*x2 + x1 - 2*x2 + 3
    return f


def grad_analitico(func, x_zero, epsilon=1e-2, max_iterations=5000, verbose=True):
  x0 = {x1: float(x_zero[0]), x2: float(x_zero[1])}
  trajetoria = [x_zero.copy()]
  grad = [sp.diff(func, var) for var in (x1, x2)]


  for i in range(max_iterations):

    grad_x0 = [float(g.subs(x0)) for g in grad]

    # Critério de parada: norma do gradiente
    if np.linalg.norm(grad_x0) < epsilon:
      return x0, i, trajetoria

    x_alpha = sp.Matrix([x0[x1], x0[x2]]) - alpha * sp.Matrix(grad_x0)

    f_alpha = func.subs({
      x1: x_alpha[0],
      x2: x_alpha[1]
    })


    # ALTERAÇÃO 1 (necessária):
    # Para funções com termos transcendentes (sin/cos/exp/log), resolver
    # simbolicamente d/dalpha = 0 costuma ser muito lento ou inviável
    # (caso da Griewank). Nesses casos pulamos o solve e usamos backtracking.
    df_alpha = sp.diff(f_alpha, alpha)
    tem_termo_transcendente = df_alpha.has(sp.sin, sp.cos, sp.exp, sp.log)

    a_val = None
    if not tem_termo_transcendente:
      try:
        alpha_star = sp.solve(df_alpha, alpha)
        reais = [s for s in alpha_star if getattr(s, "is_real", False)]
        a_val = float(reais[0]) if reais else None
      except Exception:
        a_val = None

    # Fallback numérico mínimo: backtracking no passo ao longo de -gradiente
    if a_val is None or not np.isfinite(a_val) or a_val <= 0:
      a_val = 1.0
      f0 = float(func.subs({x1: x0[x1], x2: x0[x2]}))
      grad_norm2 = float(np.dot(grad_x0, grad_x0))
      while a_val > 1e-8:
        x_test = {
          x1: x0[x1] - a_val * grad_x0[0],
          x2: x0[x2] - a_val * grad_x0[1]
        }
        f_test = float(func.subs({x1: x_test[x1], x2: x_test[x2]}))
        if f_test <= f0 - 1e-4 * a_val * grad_norm2:
          break
        a_val *= 0.5

    # ALTERAÇÃO 2 (necessária):
    # O código original usava alpha_star[0] e ignorava a_val.
    # Aqui a atualização usa SEMPRE o passo validado (a_val).
    x0_new = sp.Matrix([x0[x1], x0[x2]]) - a_val * sp.Matrix(grad_x0)
    x0 = {x1: float(x0_new[0]), x2: float(x0_new[1])}
    trajetoria.append([round(float(x0[x1]), 5), round(float(x0[x2]),5)])


    if verbose:
      print(f"ite: {i} | x = {x0} | f(x) = {round(float(func.subs({x1: x0[x1], x2: x0[x2]})), 4)}")

  # ALTERAÇÃO 3 (necessária):
  # Garante retorno mesmo se atingir o limite de iterações sem convergir.
  return x0, max_iterations, trajetoria

In [13]:
x_min, ite , traj = grad_analitico(f(), [-1,1])
print(f'x* = {x_min}')
print(f'Iterações: {ite}')

ite: 0 | x = {x1: -0.75, x2: 1.5} | f(x) = 0.375
ite: 1 | x = {x1: -1.25, x2: 1.75} | f(x) = 0.0625
ite: 2 | x = {x1: -1.125, x2: 2.0} | f(x) = -0.0938
ite: 3 | x = {x1: -1.375, x2: 2.125} | f(x) = -0.1719
ite: 4 | x = {x1: -1.3125, x2: 2.25} | f(x) = -0.2109
ite: 5 | x = {x1: -1.4375, x2: 2.3125} | f(x) = -0.2305
ite: 6 | x = {x1: -1.40625, x2: 2.375} | f(x) = -0.2402
ite: 7 | x = {x1: -1.46875, x2: 2.40625} | f(x) = -0.2451
ite: 8 | x = {x1: -1.453125, x2: 2.4375} | f(x) = -0.2476
ite: 9 | x = {x1: -1.484375, x2: 2.453125} | f(x) = -0.2488
ite: 10 | x = {x1: -1.4765625, x2: 2.46875} | f(x) = -0.2494
ite: 11 | x = {x1: -1.4921875, x2: 2.4765625} | f(x) = -0.2497
ite: 12 | x = {x1: -1.48828125, x2: 2.484375} | f(x) = -0.2498
ite: 13 | x = {x1: -1.49609375, x2: 2.48828125} | f(x) = -0.2499
ite: 14 | x = {x1: -1.494140625, x2: 2.4921875} | f(x) = -0.25
x* = {x1: -1.494140625, x2: 2.4921875}
Iterações: 15


In [14]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

def rastrigin(X):
  return 10 * len(X) + np.sum(X**2 - 10 * np.cos(2 * np.pi * X), axis=0)

def rosenbrock(X):
  return np.sum(100 * (X[1:] - X[:-1]**2)**2 + (X[:-1] - 1)**2, axis=0)

def griewank(X):
  d = X.shape[0]
  sum_ = np.sum(X**2 / 4000, axis=0)
  i = np.arange(1, d + 1).reshape(-1, *( [1] * (X.ndim - 1) ))
  cos_ = np.prod(np.cos(X / np.sqrt(i)), axis=0)
  return sum_ - cos_ + 1

def six_hump(X):
  return (4 - 2.1 * X[0]**2 + X[0]**4 / 3) * X[0]**2 + X[0] * X[1] + (-4 + 4 * X[1]**2) * X[1]**2

def sphere(X):
  return np.sum(X**2, axis=0)

def ackley(X):
  return - 20.0 * (
      np.exp(-0.2 * np.sqrt( (1/len(X)) * np.sum(X**2, axis=0) )) -
      np.exp( (1/len(X)) * np.sum(np.cos(2 * np.pi * X), axis=0))
      + 20 + np.exp(1))

def plot_grad_analitico(func, nome, trajetoria):
  path = np.array(trajetoria)

  # Grid (ajuste simples — funciona bem pra maioria)
  x_vals = np.linspace(min(path[:,0])-1, max(path[:,0])+1, 100)
  y_vals = np.linspace(min(path[:,1])-1, max(path[:,1])+1, 100)
  X, Y = np.meshgrid(x_vals, y_vals)
  Z = func(np.array([X, Y]))

  # Plot
  plt.figure(figsize=(6,5))
  plt.contour(X, Y, Z, levels=30)
  plt.plot(path[:,0], path[:,1], 'ro-', label='trajetória')

  plt.title(f"Descida do Gradiente - {nome}")
  plt.xlabel("x1")
  plt.ylabel("x2")
  plt.legend()
  plt.show()

In [3]:

x1, x2 = sp.symbols('x1 x2')
X = [x1, x2]

rastrigin = 10 * len(X) + sum(xi**2 - 10 * sp.cos(2 * sp.pi * xi) for xi in X)

rosenbrock = sum(100 * (X[i+1] - X[i]**2)**2 + (X[i] - 1)**2 for i in range(len(X) - 1))
rosenbrock = 100 * (x2 - x1**2)**2 + (x1 - 1)**2


sum_griewank = sum(xi**2 / 4000 for xi in X)
prod_griewank = 1
for i, xi in enumerate(X, start=1):
    prod_griewank *= sp.cos(xi / sp.sqrt(i))

griewank = sum_griewank - prod_griewank + 1



six_hump = (4 - 2.1 * X[0]**2 + X[0]**4 / 3) * X[0]**2 + X[0] * X[1] + (-4 + 4 * X[1]**2) * X[1]**2

sphere = sum(xi**2 for xi in X)


n = len(X)
term1 = -20.0 * sp.exp(-0.2 * sp.sqrt((1/n) * sum(xi**2 for xi in X)))
term2 = sp.exp((1/n) * sum(sp.cos(2 * sp.pi * xi) for xi in X))

ackley = term1 - term2 + 20 + sp.exp(1)

In [16]:
x_min, ite, traj = grad_analitico(rastrigin, [-1.,1.])
print(f'x* = {x_min}')
print(f'Iterações: {ite}')

ite: 0 | x = {x1: 0.0, x2: 0.0} | f(x) = 0.0
x* = {x1: 0.0, x2: 0.0}
Iterações: 1


In [4]:
# Teste focado na função griewank
x_min, ite, traj = grad_analitico(griewank, [-1.0, 1.0], epsilon=1e-3, max_iterations=300, verbose=False)
fx = float(griewank.subs({x1: x_min[x1], x2: x_min[x2]}))
print(f"[OK] griewank: x*={x_min}, f(x*)={fx:.6f}, iterações={ite}")

[OK] griewank: x*={x1: -4.197657433731067e-25, x2: 0.0016443472648188216}, f(x*)=0.000001, iterações=10
